In [ ]:
import pandas as pd
import numpy as np

import string

In [ ]:
df = pd.read_csv("books.csv")

In [ ]:
df.isna().sum()

In [ ]:
df.shape

In [ ]:
df.describe(include="object")

In [ ]:
df[df["Year"]==0]

In [ ]:
df = df[df['Year'] != 0]

In [ ]:
df["Title_lower"] = df["Title"].str.lower()
df["Title_lower"] = df["Title_lower"].str.translate(str.maketrans("", "", string.punctuation))

In [ ]:
# Wasted an hour on this code which never made out {#SAHEED_CODE}


# df["Blurb_Count"] = df.groupby(["Title", "Author", "Blurb"])["Blurb"].transform("count")

# df = df.sort_values(["Title", "Author", "Blurb_Count"], ascending=[True, True, False])
# df["Common_Blurb"] = df.groupby(["Title", "Author"])["Blurb"].transform("first")

# df["Group_Max_Count"] = df.groupby(["Title", "Author"])["Blurb_Count"].transform("max")

# mask = (df["Group_Max_Count"] > 1) & (df["Blurb"] == df["Common_Blurb"]) | (df["Group_Max_Count"] <= 1)
# df_filtered = df[mask]

# df_clean = (
#     df_filtered.sort_values("Year", ascending=False)
#     .drop_duplicates(subset=["Title", "Author"])
#     .reset_index(drop=True)
# )

# df_clean = df_clean.drop(columns=["Blurb_Count", "Common_Blurb", "Group_Max_Count"])

In [ ]:
df_clean = (
    df.sort_values("Year", ascending=False)
    .drop_duplicates(subset=["Title_lower"])
    .reset_index(drop=True)
)

df_clean

In [ ]:
df_clean["Author"] = df_clean["Author"].str.lower()
df_clean["Author"] = df_clean["Author"].str.translate(str.maketrans("", "", string.punctuation))

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(row):
    row = str(row).lower()
    row = row.translate(str.maketrans("", "", string.punctuation))
    row = " ".join(word for word in row.split() if word not in stop)
    row = " ".join(lemmatizer.lemmatize(word) for word in row.split())
    return row

df_clean['Blurb'] = df_clean['Blurb'].apply(preprocess_text)

In [ ]:
df_clean.drop(["Publisher", "Year"], axis=1)

In [ ]:
df_clean['text'] = df_clean[['Title_lower', 'Title_lower', 'Author', 'Author', 'Blurb']].agg(" ".join, axis=1)
df_clean

In [ ]:
combined_df = df_clean[['ISBN', 'Title', 'text']]
combined_df

In [ ]:
combined_df = combined_df.reset_index(drop=True)
combined_df

In [ ]:
indices_title = pd.Series(combined_df.index, index=combined_df['Title'].str.lower().str.translate(str.maketrans("", "", string.punctuation)))
indices_title

In [ ]:
indices_isbn = pd.Series(combined_df.index, index=combined_df['ISBN'])
indices_isbn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=100000, ngram_range=(1,2))
vectorized_matrix = vectorizer.fit_transform(combined_df['text'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def normalize_title(title):
    return (
        str(title)
        .strip()
        .lower()
        .translate(str.maketrans("", "", string.punctuation))
    )
def get_prediction(titles, n=12):
    selected_idx = [
        indices_title[normalize_title(title)]
        for title in titles
    ]
    user_vector = np.asarray(vectorized_matrix[selected_idx].mean(axis=0))
    sim_score = cosine_similarity(user_vector, vectorized_matrix)[0]
    sim_score[selected_idx] = -1
    sim_idx = np.argsort(sim_score)[::-1][:n]
    return combined_df["Title"].iloc[sim_idx].tolist()

In [ ]:
Pred_list = get_prediction([
    "The Trouble With Harry"
])
Pred_list